# 1. Context

This notebook analyzes OCR performance of Tesseract over synthetic generated PDF images across variois degradation levels

# 2. Imports

In [1]:
import pandas as pd
from pathlib import Path
from collections import defaultdict

In [2]:
import sys

notebook_path = Path()
sys.path.append(str(notebook_path.resolve().parent))

In [3]:
from src.viz.helper import display_box_plot

# 2. Extracted Results MetaData

In [4]:
writing_system_to_language = {'Devanagari': ['hindi','sanskrit','nepali','konkani'],
                              'tamil': ['tamil'],'telugu': ['telugu'],'Kannada': ['kannada'],'Malayalam': ['malayalam'],
                              'Bengali': ['bengali', 'assamese', 'manipuri'],'Meetei-mayek': ['manipuri'],'Gujarati': ['gujarati'],
                              'Gurmukhi': ['punjabi'],'Odia': ['oriya'],'Arabic': ['kashmiri', 'sindhi', 'urdu'],
                              'Latin': ['english'],'Ol-chiki': ['santali']}

## 2.1. Language Results Available

In [5]:
results_root = Path("../results/tesseract")

In [6]:
results_langs = [x.name.lower() for x in results_root.glob("*") if x.is_dir()]

In [7]:
language_to_writing_system = {
    "marathi": ["Devanagari"], "hindi": ["Devanagari"], "sanskrit": ["Devanagari"],
    "tamil": ["tamil"], "telugu": ["telugu"], "kannada": ["Kannada"],"malayalam": ["Malayalam"],
    "bengali": ["Bengali"], "assamese": ["Bengali"],"manipuri": ["Bengali", "Meetei-mayek"],"nepali": ["Devanagari"],
    "gujarati": ["Gujarati"], "punjabi": ["Gurmukhi"], "konkani": ["Devanagari"],"oriya": ["Odia"],"kashmiri": ["Devanagari", "Arabic"], 
    "sindhi": ["Arabic", "Devanagari"], "urdu": ["Arabic"],"english": ["Latin"], "santali": ["Ol-chiki", "Devanagari"],
    }

In [8]:
writing_sys_tessract_dict = defaultdict(list)

for language_res in results_langs:
    script = language_to_writing_system.get(language_res)[0]
    writing_sys_tessract_dict[script].append(language_res)

In [9]:
script_language_result = pd.Series(writing_sys_tessract_dict).to_frame(name='languages')
script_language_result.index.name = 'script'

In [11]:
def get_language_results_path(script: str, script_language_result: pd.DataFrame) -> list[Path]:
    """Get list of results path for given script"""

    results_script_lang = script_language_result.loc[script].to_list()[0]
    results_script_lang_path = [results_root.joinpath(lang).joinpath('results.csv') for lang in results_script_lang]

    return results_script_lang_path


# 3. Results Across Various Writing System 

## 3.1. Devanagari

In [12]:
script = 'Devanagari'
results_devanagari_path = get_language_results_path(script, script_language_result)

In [13]:
results_list = []
for result_path in results_devanagari_path:
    lang = result_path.parent.name
    df_res = pd.read_csv(result_path)
    df_res['language'] = lang
    results_list.append(df_res)

results_devanagari = pd.concat(results_list)

In [14]:
display_box_plot(results_df=results_devanagari, script='Devanagari', metric_type='CER').show()